# scANVI label transfer

**Pinned Environment:** [`envs/sc-scvi.yaml`](../../envs/sc-scvi.yaml)  

In [ ]:
from pathlib import Path
import os
import scanpy as sc
from scipy.sparse import issparse
import scvi
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import anndata as ad
from lightning.pytorch import seed_everything
import random
import torch
import sys
import session_info

In [ ]:
random.seed(0)
seed_everything(0)

scvi.settings.seed = 0
scvi.settings.num_workers = 32

In [ ]:
sys.path.append(str(Path.cwd().resolve().parents[1]))
from config.paths import BASE_DIR

adata_dir = BASE_DIR / "data/h5ad/export_02/02b_leiden"
ref_data_dir = BASE_DIR / "data" / "scrna-seq" / "h5ad" / "05_reference" # Concatenated dataset

output_dir = BASE_DIR / "data/h5ad/export_03/03a_scanvi"
scanvi_dir = BASE_DIR / 'scanvi' # csv output
scvi_dir = BASE_DIR / 'scvi/scanvi'

scanvi_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
adata = sc.read_h5ad(adata_dir / 'adata-leiden-475.h5ad')
refdata = sc.read_h5ad(ref_data_dir / 'adata-reference.h5ad')

### Prepare AnnDatas

In [ ]:
# Rename existing real "Unknown" annotation for scanvi script to work
refdata.obs["cell_label"] = (
    refdata.obs["cell_label"]
    .replace({"Unknown": "Unassigned"})
)

refdata.obs.cell_label.value_counts()

In [ ]:
adata.obs["experiment_id"] = "Xenium" # experiment_id will be used as batch key

In [ ]:
print(refdata.layers)
print(adata.layers)

In [ ]:
# restore raw counts, assign same name as query adata
#refdata.layers['counts'] = refdata.layers['raw'].copy()  # Store raw counts
#refdata.X = refdata.layers['counts'].copy()  # Set X to use raw counts

In [ ]:
adata.X = adata.layers['counts'].copy()

In [ ]:
adata.X[:5, :5].toarray()  # Convert and preview first 5x5 matrix elements

In [ ]:
print(refdata.X[:5, :5])

In [ ]:
# add output_id for refdata
#refdata.obs['output_id'] = refdata.obs['batch']
#refdata.obs.output_id.value_counts()

In [ ]:
# add column up here for filtering later only on refdata
refdata.obs['ref_data'] = 'Yes'
adata.obs['ref_data'] = 'No'

## Gene alignment and concatenation

In [ ]:
# Find the intersection of genes for integration
common_genes = list(set(adata.var_names) & set(refdata.var_names))
len(common_genes)

### Create subsets

In [ ]:
# Create copies of the datasets, keeping only common genes
adata_subset = adata[:, common_genes].copy()
refdata_subset = refdata[:, common_genes].copy()

In [ ]:
print(f"adata before subsetting: {adata.shape[1]}")
print(f"adata after subsetting: {adata_subset.shape[1]}")

print(f"refdata before subsetting: {refdata.shape[1]}")
print(f"refdata after subsetting: {refdata_subset.shape[1]}")

In [ ]:
index_match = (adata_subset.var_names == refdata_subset.var_names).all()
print(f"adata.var_names match: {index_match}")

# Concatenate

In [ ]:
print(adata_subset.obs.columns)
print(refdata_subset.obs.columns)

In [ ]:
# Concatenate datasets
adata_list = [adata_subset, refdata_subset]

adata_concat = ad.concat(adata_list, 
                  join="outer", # make sure it's outer
                  label = 'scanvi_batch',
                  index_unique="-")

In [ ]:
adata_concat # confirm number genes

In [ ]:
adata_concat.obs.output_id.value_counts()

### Perform filtering on concatenated adata

In [ ]:
sc.pp.calculate_qc_metrics(adata_concat, percent_top=(10, 20, 50, 150), inplace=True)

In [ ]:
print(f"Empty cell check (min, max): {adata_concat.X.sum(axis=1).min()}, {adata_concat.X.sum(axis=1).max()}")

12/02 - Revise scanvi logic here to not have a total_counts ceiling

In [ ]:
# or operator ensures that rows satisfying at least one condition are kept
# so if adata.obs.ref_data = No (study samples), then the cell is kept
# OR if adata.obs.ref_data = Yes AND the cell has acceptable total counts the row is also kept

adata_concat = adata_concat[
    (adata_concat.obs["ref_data"] == "No") | # Keep all non-reference samples
    (
        (adata_concat.obs["ref_data"] == "Yes") & # Apply total_counts filter only to reference samples
        (adata_concat.obs["total_counts"] > 50)
    )
].copy()

In [ ]:
print(f"Empty cell check (min, max): {adata_concat.X.sum(axis=1).min()}, {adata_concat.X.sum(axis=1).max()}")

# Examine Batch Effects

In [ ]:
adata_concat.X[:5, :5].toarray()

In [ ]:
# Normalize and run PCA
sc.pp.normalize_total(adata_concat)
sc.pp.log1p(adata_concat)
sc.pp.pca(adata_concat, n_comps = 10, random_state = 0) # added arguments 12/2 to reduce PCA calculation for better visualization

# UMAP colored by output_id
sc.pp.neighbors(adata_concat, n_pcs = 10)
sc.tl.umap(adata_concat)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot UMAP by batch identifier
sc.pl.umap(adata_concat, color="experiment_id", show=False, ax=axes[0])
axes[0].set_title("UMAP Colored by Batch")

# Plot UMAP by ref_data
sc.pl.umap(adata_concat, color="ref_data", show=False, ax=axes[1])
axes[1].set_title("UMAP Colored by Reference Data (Yes/No)")

plt.show()
plt.close()

## Train scVI model

In [ ]:
# Check batch key
adata_concat.obs["experiment_id"] = adata_concat.obs["experiment_id"].astype("category")
adata_concat.obs["experiment_id"] = pd.Categorical(adata_concat.obs["experiment_id"])
adata_concat.obs['experiment_id'].value_counts()

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata_concat, 
    layer="counts", # use raw count data
    batch_key='experiment_id', # biggest batch effect is panel; 12/02 
)

model = scvi.model.SCVI(adata_concat, gene_likelihood="zinb") # gene_likelihood zinb added 12/2
print('starting model training')

model.train(early_stopping=True, 
            enable_progress_bar=True,
            accelerator = 'gpu'
           )

In [ ]:
model.save(os.path.join(scvi_dir, '02_model') , prefix='02_label_transfer_')

In [ ]:
SCVI_LATENT_KEY = "X_scVI_refalign"
adata_concat.obsm[SCVI_LATENT_KEY] = model.get_latent_representation()

### Compute neighbors graph and UMAP

In [ ]:
# neighbors
sc.pp.neighbors(adata_concat, use_rep="X_scVI_refalign", key_added='neighbors_scvi_refalign')
sc.tl.umap(adata_concat, neighbors_key='neighbors_scvi_refalign')

In [ ]:
sc.pl.umap(adata_concat, color = ['experiment_id', 'cell_label'], frameon = False)

### Export refaligned adata

In [ ]:
# save umap dims to csv
umap_df = pd.DataFrame(adata_concat.obsm['X_umap'], index=adata_concat.obs_names, columns=['umap1', 'umap2'])
filename = os.path.join(scanvi_dir, 'adata_scvi_refalign_umap_coords.csv')
umap_df.to_csv(filename)

In [ ]:
filename = os.path.join(output_dir, 'adata_concat_scvi_refalign.h5ad')
os.makedirs(os.path.dirname(filename), exist_ok = True)

adata_concat.write_h5ad(filename, compression='gzip')

### Prep data for scANVI

In [ ]:
# Add 'Unknown' category for original samples if missing, then fill NaN
if 'Unknown' not in adata_concat.obs['cell_label'].cat.categories:
    adata_concat.obs['cell_label'] = adata_concat.obs['cell_label'].cat.add_categories('Unknown')

adata_concat.obs = adata_concat.obs.fillna(value={'cell_label': 'Unknown'})

In [ ]:
adata_concat.obs['cell_label'].value_counts() # confirm that cell_label is the most granular

Below is important for mapping labels in the unintegrated original adata

In [ ]:
# Subset only query dataset (unlabeled before scANVI training)
query_cells = adata_concat.obs["cell_label"] == "Unknown"
query_cells.to_csv(os.path.join(scanvi_dir, 'query_cells-subtype.csv'))
query_cells

In [ ]:
# fix barcodes; suffix was added during concat
query_cells_copy = query_cells.copy()
query_cells_copy.index = query_cells_copy.index.map(lambda x: x[:-2])
query_cells_copy

## Run scANVI

In [ ]:
# scanvi label transfer
scanvi_model = scvi.model.SCANVI.from_scvi_model(model, 
                                                 adata = adata_concat, 
                                                 unlabeled_category = 'Unknown', # Entries in labels_key to label
                                                 labels_key = 'cell_label') # Column to transfer labels from

#scanvi_model.train(max_epochs=20, 
#                   accelerator = 'gpu',
#                   n_samples_per_label = None) # this is longer training because it's not subsampling

In [ ]:
# Edited 12/02
scanvi_model.train(accelerator = 'gpu',
                   n_samples_per_label = None) # this is longer training because it's not subsampling

In [ ]:
# save scANVI model
scanvi_model.save(os.path.join(scvi_dir, '03_model_scanvi') , prefix='03_scanvi_')

### Reference mapping step

In [ ]:
SCANVI_LATENT_KEY = "X_scANVI"
SCANVI_PREDICTION_KEY = "scanvi_labels"

adata_concat.obsm[SCANVI_LATENT_KEY] = scanvi_model.get_latent_representation(adata_concat)
adata_concat.obs[SCANVI_PREDICTION_KEY] = scanvi_model.predict(adata_concat) # this fills out the labels

In [ ]:
adata_concat.obs['scanvi_labels'].value_counts()

In [ ]:
filename = os.path.join(output_dir, 'adata_concat-scvi-scanvi-predictions.h5ad')
os.makedirs(os.path.dirname(filename), exist_ok = True)

adata_concat.write_h5ad(filename, compression='gzip')

## Map query labels onto original adata

In [ ]:
# re-read fresh adata with all the genes, this is the query
adata_query = sc.read_h5ad(os.path.join(adata_dir, 'adata-leiden-475.h5ad'))
adata_query.obs.head()

In [ ]:
# create adata_labeled from adata_concat which has the labels
adata_labeled = adata_concat[adata_concat.obs['scanvi_batch'] == '0'].copy() # batch is stored as string

print(adata_labeled.obs['scanvi_batch'].value_counts())
print('')
print(adata_labeled.obs['output_id'].value_counts())

In [ ]:
# Remove suffix from labeled adata subset from adata_concat
adata_labeled.obs.index = adata_labeled.obs.index.astype(str)
adata_labeled.obs.index = adata_labeled.obs.index.str[:-2]
adata_labeled.obs.head()

In [ ]:
# fix barcodes; suffix was added during concat
#query_cells_copy = query_cells.copy()
#query_cells_copy.index = query_cells_copy.index.map(lambda x: x[:-2])

In [ ]:
#  Clean up query_cells_copy
## check it's a list of valid cell IDs that exist in both AnnData objects

if isinstance(query_cells_copy, pd.Series) and query_cells_copy.dtype == bool:
    query_cells_copy = query_cells_copy[query_cells_copy].index.tolist()

# Restrict to intersection of cell IDs in both datasets
query_cells_copy = [c for c in query_cells_copy if c in adata_labeled.obs_names and c in adata_query.obs_names]

print(f"{len(query_cells_copy)} overlapping cells between labeled and query datasets.")

In [ ]:
# Transfer predictions back to the original query dataset
adata_query.obs.loc[query_cells_copy, 'scanvi_labels'] = adata_labeled.obs.loc[query_cells_copy, "scanvi_labels"]
adata_query.obs['scanvi_labels']

In [ ]:
adata_query.obs.scanvi_labels.value_counts()

## Export labeled adata

In [ ]:
h5ad_path = os.path.join(output_dir, 'adata-scanvi-labels.h5ad')
adata_query.write_h5ad(h5ad_path, compression='gzip')

print(h5ad_path)